In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import pandas as pd
from constants import MAX_SEQ_LEN, DATOS_PATH
from collections import Counter

In [4]:
df = pd.read_csv(DATOS_PATH, sep="\t", header=None, encoding="utf-8")
df = df.drop(columns=[0, 2])
df = df.rename(columns={1: "spanish", 3: "english"})
df.head()

,spanish,english
0,¡Intentemos algo!,Let's try something.
1,Tengo que irme a dormir.,I have to go to sleep.
2,¿Qué estás haciendo?,What are you doing?
3,¿Qué estás haciendo?,What are you doing now?
4,¿Qué estás haciendo?,What are you up to?


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 279327 entries, 0 to 279326
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   spanish  279327 non-null  str  
 1   english  279326 non-null  str  
dtypes: str(2)
memory usage: 4.3 MB


### Traductor

In [6]:
spanish = df["spanish"].tolist()
english = df["english"].tolist()

In [7]:
print(spanish[:3])
print(english[:3])

['¡Intentemos algo!', 'Tengo que irme a dormir.', '¿Qué estás haciendo?']
["Let's try something.", 'I have to go to sleep.', 'What are you doing?']


In [8]:
def preprocesar_enunciado(enunciado: str):
    enunciado = str(enunciado)
    enunciado = enunciado.lower().strip()
    # enunciado = enunciado.replace(".", " .")
    # enunciado = enunciado.replace(",", " ,")
    # enunciado = enunciado.replace("¿", " ¿ ")
    # enunciado = enunciado.replace("?", " ? ")
    # enunciado = enunciado.replace("¡", " ¡ ")
    # enunciado = enunciado.replace("!", " ! ")
    enunciado = '<sos> ' + enunciado + ' <eos>'
    return enunciado

In [9]:
enunciado = "¡Hola! ¿Cómo estás?"
print(preprocesar_enunciado(enunciado))

<sos> ¡hola! ¿cómo estás? <eos>


In [10]:
spanish_preprocesado = [preprocesar_enunciado(s) for s in spanish]
english_preprocesado = [preprocesar_enunciado(e) for e in english]

In [11]:
print(spanish_preprocesado[0:3])
print(english_preprocesado[0:3])

['<sos> ¡intentemos algo! <eos>', '<sos> tengo que irme a dormir. <eos>', '<sos> ¿qué estás haciendo? <eos>']
["<sos> let's try something. <eos>", '<sos> i have to go to sleep. <eos>', '<sos> what are you doing? <eos>']


In [12]:
def construir_palabras(enunciados):
    palabras = [palabra for enunciado in enunciados for palabra in enunciado.split(" ")]
    palabras_count = Counter(palabras)
    palabras_ordenadas = sorted(palabras_count.items(), key=lambda x: x[1], reverse=True)
    palabra_idx = {palabra: idx for idx, (palabra, _) in enumerate(palabras_ordenadas, 2)}
    palabra_idx["<pad>"] = 0
    palabra_idx["<unk>"] = 1
    
    idx_palabra = {idx: palabra for palabra, idx in palabra_idx.items()}
    return palabra_idx, idx_palabra

In [13]:
english_palabra_idx, english_idx_palabra = construir_palabras(english_preprocesado)
spanish_palabra_idx, spanish_idx_palabra = construir_palabras(spanish_preprocesado)

In [16]:
spanish_vocab_size = len(spanish_palabra_idx)
english_vocab_size = len(english_palabra_idx)

spanish_vocab_size, english_vocab_size

(90647, 57709)